# Tara N1 Pretraining

- **Data set:** 500M token subset of FineWeb
- **model.py:** `https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/model.py`
- **train_utils.py:** `https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/train_utils.py`

In [3]:
import torch
from torch import nn
import tiktoken
import requests
import os

tokenizer = tiktoken.get_encoding("gpt2")
device = "gpu" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [ ]:
model_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/model.py")
with open("model.py", "w") as f:
    f.write(model_res.text)
print("Downloaded model.py successfully.")


train_utils_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/train_utils.py")
with open("train_utils.py", "w") as f:
    f.write(train_utils_res.text)
print("Downloaded train_utils.py successfully.")

In [7]:
from model import *
from train_utils import *

In [5]:
config = GPTConfig(
    vocab_size=tokenizer.n_vocab,
    block_size=1024,
    d_model=512,
    hidden_layers=2048,
    n_heads=8,
    n_layers=12,
)

In [ ]:
modelV1 = CustomGPT(config).to(device)

calc_params(modelV1)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(modelV1.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler()

Total Parameters: 89,867,345
Trainable Parameters: 89,867,345


# The Dataset

In [ ]:
from datasets import load_dataset
from tqdm.auto import tqdm
target_tokens = 1_000_000_000
ds = load_dataset("HuggingFaceFW/fineweb", split="train", name="sample-10B", streaming=True)

tokens = []
pbar = tqdm(total=target_tokens)
for sample in ds:
    text = sample["text"]
    tokenized = tokenizer.encode(text)
    tokens.extend(tokenized)
    curr = min(len(tokens), target_tokens)
    pbar.n = curr
    pbar.update(0)
    if len(tokens) >= target_tokens:
        break

pbar.close()
print(f"Collected {len(tokens)} tokens.")


In [ ]:
data = torch.tensor(tokens, dtype=torch.long)

train_split = int(0.9 * len(data))
train_dataloader, test_dataloader = create_dataloaders(data, train_split, device, block_size = config.block_size, batch_size=32)

# Pretraining the model

In [ ]:
from tqdm.auto import tqdm
steps = 31000
train_iter = iter(train_dataloader)
for step in tqdm(range(steps)):
    modelV1.train()
    train_loss = train_step(modelV1, train_dataloader, train_iter, loss_fn, optimizer, scaler, device)
    
    if step % 5000 == 0:
        modelV1.eval()
        test_loss = test_step(modelV1, test_dataloader, loss_fn, device)
        print(f"Step {step} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f}")

In [ ]:
torch.save(modelV1.state_dict(), "tara_n1_pretrain_v1.pth")

# Testing



In [ ]:
test_model = CustomGPT(config).to(device)
state_dict = torch.load("tara_n1_pretrain_v1.pth", map_location=device)
state_dict = {k.removeprefix('module.'): v for k, v in state_dict.items()} 
test_model.load_state_dict(state_dict)

In [ ]:
query = "Once upon a time, "

context = torch.tensor(tokenizer.encode(query), dtype=torch.long).unsqueeze(0).to(device)

test_model.eval()
with torch.inference_mode():
    output = test_model.generate(context, max_new_tokens=100)

print(f"Input:\n{query}\n")
print(f"Output:\n{tokenizer.decode(output[0].tolist())}")
